In [2]:
import pandas as pd
import folium
from folium.plugins import HeatMap, MarkerCluster
import plotly.express as px


In [3]:
# ---------- Sample earthquake dataset ----------

quake_df = pd.DataFrame({
    'place': [
        'Ambon, Indonesia',
        'Antofagasta, Chile',
        'Kermadec Islands, NZ',
        'Anchorage, USA',
        'Izmir, Turkey',
        'Mindanao, Philippines',
        'Fiji Islands',
        'Ridgecrest, USA',
        'Sumatra, Indonesia',
        'Kathmandu, Nepal'
    ],

    'lat': [
        -3.70, -23.65, -29.23,
        61.22, 38.42, 6.97,
        -17.71, 35.61, -0.79, 27.71
    ],

    'lon': [
        128.18, -70.40, -177.85,
        -149.90, 27.13, 122.08,
        178.07, -117.60, 100.36, 85.32
    ],

    'magnitude': [
        5.8, 6.1, 5.4, 4.9, 6.6,
        5.2, 5.9, 6.4, 6.0, 7.8
    ]
})

In [4]:
# ---------- 1. Folium Marker Map ----------

m_markers = folium.Map(
    location=[10, 60],
    zoom_start=2,
    tiles='CartoDB positron'
)

cluster = MarkerCluster().add_to(
    m_markers
)

for _, row in quake_df.iterrows():

    folium.CircleMarker(
        location=[
            row['lat'],
            row['lon']
        ],
        radius=4 + row['magnitude'] * 1.5,
        color='crimson',
        fill=True,
        fill_opacity=0.7,
        tooltip=(
            f"{row['place']} - "
            f"M{row['magnitude']}"
        ),
        popup=folium.Popup(
            f"<b>{row['place']}</b><br>"
            f"Magnitude: {row['magnitude']}",
            max_width=220
        )
    ).add_to(cluster)

folium.LayerControl().add_to(
    m_markers
)

m_markers.save(
    'earthquake_markers.html'
)

/usr/local/lib/python3.13/dist-packages/folium/raster_layers.py:130: UserWarning: CartoDB tiles now require an API key. Please provide one to continue using the tiles. You can request the key at https://carto.com/basemaps/apikey/.
  tiles = tiles.build_url(fill_subdomain=False, scale_factor="{r}")  # type: ignore


In [5]:
# ---------- 2. Folium Heatmap ----------

m_heat = folium.Map(
    location=[10, 60],
    zoom_start=2,
    tiles='CartoDB dark_matter'
)

heat_data = (
    quake_df[
        ['lat', 'lon', 'magnitude']
    ].values.tolist()
)

HeatMap(
    heat_data,
    radius=25,
    blur=15,
    max_zoom=4
).add_to(m_heat)

m_heat.save(
    'earthquake_heatmap.html'
)

In [6]:
# ---------- 3. Plotly Scatter Map ----------

fig_scatter = px.scatter_mapbox(
    quake_df,
    lat='lat',
    lon='lon',
    size='magnitude',
    color='magnitude',
    hover_name='place',
    hover_data={
        'magnitude': True,
        'lat': False,
        'lon': False
    },
    color_continuous_scale='YlOrRd',
    size_max=20,
    zoom=1,
    mapbox_style='carto-positron',
    title='Recent Earthquakes - Magnitude by Location'
)

fig_scatter.update_layout(
    margin={
        'r': 0,
        't': 40,
        'l': 0,
        'b': 0
    }
)

fig_scatter.show()

In [7]:
# ---------- 4. Plotly Density Map ----------

fig_density = px.density_mapbox(
    quake_df,
    lat='lat',
    lon='lon',
    z='magnitude',
    radius=35,
    zoom=1,
    mapbox_style='carto-positron',
    color_continuous_scale='Inferno',
    title='Earthquake Density Weighted by Magnitude'
)

fig_density.update_layout(
    margin={
        'r': 0,
        't': 40,
        'l': 0,
        'b': 0
    }
)

fig_density.show()

In [8]:
# ---------- 5. Choropleth Map ----------

country_df = pd.DataFrame({
    'country': [
        'Indonesia', 'Chile', 'New Zealand',
        'United States', 'Turkey',
        'Philippines', 'Fiji', 'Nepal'
    ],

    'iso_alpha': [
        'IDN', 'CHL', 'NZL', 'USA',
        'TUR', 'PHL', 'FJI', 'NPL'
    ],

    'quake_count_2024': [
        312, 198, 87, 145,
        76, 210, 54, 41
    ]
})

fig_choropleth = px.choropleth(
    country_df,
    locations='iso_alpha',
    color='quake_count_2024',
    hover_name='country',
    color_continuous_scale='Viridis',
    title='Earthquake Count by Country (Sample Data, 2024)'
)

fig_choropleth.update_geos(
    showcoastlines=True,
    showland=True
)

fig_choropleth.show()